# B1-H2 L4 — Remaining validation inference

This notebook continues inference on every original B1-H2 validation prompt except the exact 100 already run in the L4 pilot. It does not modify the frozen prompt set. Results are validation-only; freeze the threshold only after the full validation set is complete and reviewed. The model and tokenizer hashes, context and first-token scoring follow the registered Kaggle configuration.

Before running: in Runtime → Change runtime type select **Runtime Version 2026.10 (Python 3.13.x)** and **GPU L4**, then reconnect. Use Colab Runtime Version 2026.10 with Python 3.13.x; record the exact Python patch version because Colab may report a different patch from Kaggle (3.13.15). Kaggle recorded NumPy 2.1.3. The setup cell checks Python and GPU before mounting Drive or installing anything. After any runtime reconnect, run cells from the top: the dependency cell compiles and installs `llama-cpp-python` for CUDA before the model-load cell imports it. This notebook downloads the original Q4_K_M checkpoint, checks its SHA-256, builds `llama-cpp-python==0.3.16` for L4 (SM 8.9), captures native load/offload logs, and stores per-row `logprob_0`/`logprob_1`. If L4 cannot fully offload or load, use Runtime → Change runtime type and choose A100 or G4, keeping Runtime Version 2026.10, then rerun the notebook.

Bundle đã được tải lên Google Drive: [b1h2_l4_remaining_validation_bundle.zip](https://drive.google.com/file/d/1LWGkH3tmNgWml8s_JTR7A-0ynYb0__L_/view?usp=drivesdk). Notebook đọc file từ `My Drive/GRACE_B1H2_COLAB_L4_REMAINING_20261106/` và ghi log/kết quả vào thư mục con `l4_remaining_validation_output/`.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, zipfile, subprocess, platform, time, re, shutil, math, contextlib, io, importlib.metadata
from datetime import datetime, timezone

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f'Requires Colab Runtime Version 2026.10 / Python 3.13.x to match the Kaggle Python 3.13 runtime family; got {sys.version.split()[0]}. In Runtime → Change runtime type choose Runtime Version 2026.10, then reconnect and run from the top.')
_nvidia_smi = shutil.which('nvidia-smi') or ('/usr/local/cuda/bin/nvidia-smi' if Path('/usr/local/cuda/bin/nvidia-smi').is_file() else None)
if _nvidia_smi:
    _preflight_gpu = subprocess.run([_nvidia_smi, '--query-gpu=name', '--format=csv,noheader'], check=True, text=True, capture_output=True).stdout.strip()
else:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError('Cannot detect GPU: nvidia-smi is missing and PyTorch is unavailable. Select Runtime → Change runtime type → GPU L4, reconnect, then run again.') from exc
    if not torch.cuda.is_available():
        raise RuntimeError('Colab has not attached a CUDA GPU (nvidia-smi missing and torch.cuda.is_available() is False). Select GPU L4 in Runtime → Change runtime type, reconnect, then run again.')
    _preflight_gpu = torch.cuda.get_device_name(0)
if 'L4' not in _preflight_gpu.upper():
    raise RuntimeError(f'This pilot is configured for L4; selected runtime reports {_preflight_gpu!r}. Select GPU L4 and reconnect.')
print(f'Preflight passed: Python {sys.version.split()[0]}, GPU {_preflight_gpu}')

WORK = Path('/content/b1h2_l4_remaining_validation')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_DIR = Path('/content/drive/MyDrive/GRACE_B1H2_COLAB_L4_REMAINING_20261106')
BUNDLE_DRIVE_URL = 'https://drive.google.com/file/d/1FuYcMkEuMBAFYexmg0c1HL5BxPB_L_Vg/view?usp=drivesdk'
BUNDLE_PATH = DRIVE_DIR / 'b1h2_l4_remaining_validation_bundle.zip'
OUT = DRIVE_DIR / 'l4_remaining_validation_output'
EVENTS = OUT / 'events.log'
PILOT_START = time.monotonic()
MAX_RUNTIME_SECONDS = 9 * 60 * 60

def note(msg):
    line = f'[{datetime.now(timezone.utc).isoformat()}] {msg}'
    print(line, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f: f.write(line + '\n')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Download the bundle from {BUNDLE_DRIVE_URL} into {BUNDLE_PATH}')
note(f'Bundle: {BUNDLE_PATH}; output: {OUT}')


In [ ]:
# Record runtime facts; nvidia-smi is optional if torch can identify the assigned GPU.
_nvidia_smi = shutil.which('nvidia-smi') or ('/usr/local/cuda/bin/nvidia-smi' if Path('/usr/local/cuda/bin/nvidia-smi').is_file() else None)
if _nvidia_smi:
    gpu = subprocess.run([_nvidia_smi, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True, text=True, capture_output=True).stdout.strip()
    (OUT / 'runtime_nvidia_smi.txt').write_text(subprocess.run([_nvidia_smi], check=True, text=True, capture_output=True).stdout, encoding='utf-8')
else:
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU unavailable; reconnect with GPU L4 selected.')
    props = torch.cuda.get_device_properties(0)
    gpu = f'{torch.cuda.get_device_name(0)}, {props.total_memory / (1024**2):.0f} MiB; nvidia-smi unavailable'
    (OUT / 'runtime_nvidia_smi.txt').write_text(gpu + '\n', encoding='utf-8')
if 'L4' not in gpu.upper(): raise RuntimeError(f'Expected L4, got {gpu!r}; select GPU L4 and reconnect.')
print(gpu)
_nvcc = shutil.which('nvcc') or ('/usr/local/cuda/bin/nvcc' if Path('/usr/local/cuda/bin/nvcc').is_file() else None)
if not _nvcc: raise RuntimeError('CUDA GPU found but nvcc is missing. Reconnect to a Colab GPU runtime with the CUDA toolkit, then rerun.')
for cmd, name in [([_nvcc, '--version'], 'nvcc_version.txt'), ([sys.executable, '--version'], 'python_version.txt')]:
    p = subprocess.run(cmd, text=True, capture_output=True)
    (OUT / name).write_text(p.stdout + p.stderr, encoding='utf-8')
try:
    memory_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
except (AttributeError, OSError, ValueError):
    memory_gib = None
if memory_gib is not None:
    (OUT / 'system_memory_gib.txt').write_text(f'{memory_gib:.2f} GiB\n', encoding='utf-8')
    print(f'System RAM: {memory_gib:.2f} GiB')
note(f'Confirmed Python {sys.version.split()[0]} and L4 runtime; GPU compute capability for this build is SM 8.9.')


In [ ]:
# Pin dependencies and build llama.cpp with logs on Colab's local disk.
# Google Drive FUSE can disconnect during a long CUDA build, so never stream compiler output to OUT.
LOCAL_LOG_DIR = Path('/content/b1h2_l4_setup_logs')
LOCAL_LOG_DIR.mkdir(parents=True, exist_ok=True)

def sync_setup_log(name):
    source = LOCAL_LOG_DIR / name
    try:
        OUT.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, OUT / name)
        print(f'Saved setup log to Drive: {OUT / name}', flush=True)
    except OSError as exc:
        print(f'Could not copy {name} to Drive ({exc}); local copy remains at {source}', flush=True)

packages = [
    'numpy==2.1.3',
    'huggingface-hub==0.36.2',
    'tokenizers==0.22.2',
    'transformers==4.57.1',
    'scikit-learn==1.5.2',
]
name = 'install_python_dependencies.log'
with (LOCAL_LOG_DIR / name).open('w', encoding='utf-8') as log:
    p = subprocess.Popen([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    line_count = 0
    for line in p.stdout:
        log.write(line); line_count += 1
        if line_count % 50 == 0: print(f'Python dependencies: captured {line_count} lines', flush=True)
    result = p.wait()
sync_setup_log(name)
if result != 0: raise RuntimeError(f'Pinned Python dependency installation failed; local log: {LOCAL_LOG_DIR / name}')

os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=89'
os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
os.environ['FORCE_CMAKE'] = '1'
name = 'install_llama_cpp_cuda.log'
with (LOCAL_LOG_DIR / name).open('w', encoding='utf-8') as log:
    cmd = [sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-binary', 'llama-cpp-python', 'llama-cpp-python==0.3.16']
    log.write(f'Command: {cmd}\nCMAKE_ARGS={os.environ["CMAKE_ARGS"]}\n'); log.flush()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
    line_count = 0
    for line in p.stdout:
        log.write(line); line_count += 1
        if line_count % 100 == 0: print(f'CUDA build: captured {line_count} lines', flush=True)
    result = p.wait()
sync_setup_log(name)
if result != 0: raise RuntimeError(f'CUDA build of llama-cpp-python failed; local log: {LOCAL_LOG_DIR / name}')


In [ ]:
# Verify archive, source prompts, fixed sample IDs, and registered checkpoint identity.
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''): h.update(b)
    return h.hexdigest()

with zipfile.ZipFile(BUNDLE_PATH) as z:
    bundle_manifest = json.loads(z.read('bundle_manifest.json'))
    prompt_bytes = z.read('prompts.jsonl')
if hashlib.sha256(prompt_bytes).hexdigest() != bundle_manifest['prompts_sha256']:
    raise ValueError('Prompt bundle hash mismatch')
rows = [json.loads(line) for line in prompt_bytes.decode('utf-8').splitlines() if line.strip()]
if len(rows) != bundle_manifest['count'] or len({r['id'] for r in rows}) != bundle_manifest['count']:
    raise ValueError('Expected 100 unique pilot prompts')
if bundle_manifest['source_archive_sha256'] != '9db0d3eebb66a2f83c1f9803ceb0192954a3dfe2c3de9515188a996ede84ed7b':
    raise ValueError('Unexpected source archive hash')
if bundle_manifest['prompts_sha256'] != 'e4f2765732c1c02c344ca9839cd2230e696f24079c7a54ccf4e7c42fbab09fb7':
    raise ValueError('Unexpected prompt hash')
for row in rows:
    if hashlib.sha256(row['prompt'].encode('utf-8')).hexdigest() != row['prompt_sha256']:
        raise ValueError(f'Prompt hash mismatch: {row["id"]}')
if {d: sum(r['dataset'] == d for r in rows) for d in bundle_manifest['dataset_counts']} != bundle_manifest['dataset_counts']:
    raise ValueError('Pilot sample must contain exactly 50 prompts per dataset')
note(f'Bundle verified: {len(rows)} prompts; prompts_sha256={bundle_manifest["prompts_sha256"]}')


In [ ]:
# Download the registered GGUF and refuse a changed artifact.
from huggingface_hub import hf_hub_download
MODEL_REPO = 'ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF'
MODEL_FILE = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'
TOKENIZER_REPO = 'Qwen/Qwen3-Coder-30B-A3B-Instruct'
TOKENIZER_REVISION = 'b2cff646eb4bb1d68355c01b18ae02e7cf42d120'
model_path = Path(hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, revision='main'))
model_digest = sha256_file(model_path)
if model_digest != MODEL_SHA256:
    raise ValueError(f'GGUF SHA-256 mismatch: expected {MODEL_SHA256}, got {model_digest}')
note(f'Model verified: {model_path}; sha256={model_digest}; revision=local-sha256:{model_digest}')


In [ ]:
# Capture native C/C++ stdout and stderr so the llama.cpp layer/buffer offload report is retained.
@contextlib.contextmanager
def capture_native_fds(path):
    path = Path(path)
    with path.open('w', encoding='utf-8') as f:
        sys.stdout.flush(); sys.stderr.flush()
        saved = [os.dup(1), os.dup(2)]
        try:
            os.dup2(f.fileno(), 1); os.dup2(f.fileno(), 2)
            yield
        finally:
            sys.stdout.flush(); sys.stderr.flush()
            os.dup2(saved[0], 1); os.dup2(saved[1], 2)
            os.close(saved[0]); os.close(saved[1])

try:
    _llama_cpp_installed = importlib.metadata.version('llama-cpp-python')
except importlib.metadata.PackageNotFoundError as exc:
    raise RuntimeError('llama-cpp-python is not installed in this runtime. Run the earlier dependency/build cell (the one that compiles CUDA for L4) to completion, then rerun this cell. A runtime reconnect clears installed packages.') from exc
try:
    _llama_cpp_installed = importlib.metadata.version('llama-cpp-python')
except importlib.metadata.PackageNotFoundError as exc:
    raise RuntimeError('llama-cpp-python is not installed in this runtime. Run the earlier dependency/build cell (the one that compiles CUDA for L4) to completion, then rerun this cell. A runtime reconnect clears installed packages.') from exc
from transformers import AutoTokenizer
from llama_cpp import Llama, __version__ as llama_cpp_version, llama_supports_gpu_offload
if llama_cpp_version != '0.3.16': raise RuntimeError(f'Unexpected llama-cpp-python {llama_cpp_version}')
if not llama_supports_gpu_offload(): raise RuntimeError('CUDA offload support is unavailable')
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_REPO, revision=TOKENIZER_REVISION)
label_ids = []
for label in ('0','1'):
    ids = tokenizer.encode(label, add_special_tokens=False)
    if len(ids) != 1: raise ValueError(f'Label {label} is not one token: {ids}')
    label_ids.append(ids[0])

load_started = time.monotonic()
load_error = None
llm = None
try:
    with capture_native_fds(OUT / 'llama_cpp_model_load_offload.log'):
        llm = Llama(model_path=str(model_path), n_ctx=8192, n_batch=256, n_gpu_layers=-1, logits_all=True, seed=42, verbose=True)
except Exception as e:
    load_error = repr(e)
    note(f'Model load failed: {load_error}')
if llm is None:
    summary = {'status':'load_failed','error':load_error,'gpu':gpu,'model_sha256':model_digest,'llama_cpp_python':llama_cpp_version,'context_tokens':8192,'gpu_layers_requested':-1,'load_seconds':time.monotonic()-load_started}
    (OUT / 'load_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
    raise RuntimeError('L4 could not load the model; inspect llama_cpp_model_load_offload.log, then retry on A100/G4.')

for label, hf_id in zip(('0','1'), label_ids):
    gguf_ids = llm.tokenize(label.encode(), add_bos=False, special=False)
    if gguf_ids != [hf_id]: raise ValueError(f'Label tokenizer mismatch for {label}: HF={hf_id}, GGUF={gguf_ids}')

load_log = (OUT / 'llama_cpp_model_load_offload.log').read_text(encoding='utf-8', errors='replace')
offload_lines = [line.strip() for line in load_log.splitlines() if 'offload' in line.lower() or 'buffer size' in line.lower()]
layer_reports = re.findall(r'offloaded\s+(\d+)\s*/\s*(\d+)\s+layers?\s+to\s+GPU', load_log, flags=re.I)
all_layers_offloaded = bool(layer_reports) and any(int(a) >= int(b) for a,b in layer_reports)
summary = {'status':'loaded','gpu':gpu,'model_sha256':model_digest,'model_repo':MODEL_REPO,'model_revision':'local-sha256:'+model_digest,'quantization':'Q4_K_M','llama_cpp_python':llama_cpp_version,'transformers':importlib.metadata.version('transformers'),'huggingface_hub':importlib.metadata.version('huggingface-hub'),'tokenizers':importlib.metadata.version('tokenizers'),'numpy':importlib.metadata.version('numpy'),'tokenizer':TOKENIZER_REPO,'tokenizer_revision':TOKENIZER_REVISION,'context_tokens':8192,'n_batch':256,'gpu_layers_requested':-1,'logits_all':True,'layer_reports':layer_reports,'all_layers_offloaded':all_layers_offloaded,'offload_buffer_lines':offload_lines,'load_seconds':time.monotonic()-load_started,'source_archive_sha256':bundle_manifest['source_archive_sha256'],'bundle_sha256':sha256_file(BUNDLE_PATH),'bundle_drive_url':BUNDLE_DRIVE_URL,'prompts_sha256':bundle_manifest['prompts_sha256'],'n_prompts':len(rows),'validation_continuation':True,'pilot_only':False,'validation_only':True}
(OUT / 'load_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
note(f'Model loaded; all_layers_offloaded={all_layers_offloaded}; layer_reports={layer_reports}')


In [ ]:
# Resume-safe inference over all remaining validation prompts; one record is fsynced after each completed prompt.
import numpy as np

def binary_scores(logits, ids):
    values = np.asarray(logits, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    m = float(np.max(values))
    normalizer = m + float(np.log(np.exp(values - m).sum()))
    lp0 = float(values[ids[0]] - normalizer)
    lp1 = float(values[ids[1]] - normalizer)
    s = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))
    top = np.argsort(values)[-5:][::-1]
    return {
        'logprob_0': lp0, 'logprob_1': lp1,
        'p0': math.exp(lp0), 'p1': math.exp(lp1), 's': s,
        'pred_argmax': int(s >= 0.5),
        'greedy_token_id': int(np.argmax(values)),
        'top5': [{'token_id': int(i), 'logprob': float(values[i] - normalizer)} for i in top],
    }

records_path = OUT / 'records.jsonl'
done = {}
if records_path.exists():
    for line in records_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            r = json.loads(line)
            if r['id'] in done:
                raise ValueError(f'Duplicate checkpoint ID: {r["id"]}')
            done[r['id']] = r

with records_path.open('a', encoding='utf-8', newline='\n') as f:
    for ix, item in enumerate(rows, 1):
        if time.monotonic() - PILOT_START >= MAX_RUNTIME_SECONDS:
            note('Reached the 9-hour safety limit; completed records are preserved for resume.')
            break
        if item['id'] in done:
            if done[item['id']]['prompt_sha256'] != item['prompt_sha256']:
                raise ValueError(f'Checkpoint prompt mismatch: {item["id"]}')
            continue
        if hashlib.sha256(item['prompt'].encode('utf-8')).hexdigest() != item['prompt_sha256']:
            raise ValueError(f'Prompt hash mismatch: {item["id"]}')
        chat = tokenizer.apply_chat_template([{'role':'user','content':item['prompt']}], tokenize=False, add_generation_prompt=True)
        hf_ids = tokenizer.apply_chat_template([{'role':'user','content':item['prompt']}], tokenize=True, add_generation_prompt=True)
        gguf_ids = llm.tokenize(chat.encode('utf-8'), add_bos=False, special=True)
        if list(hf_ids) != gguf_ids:
            raise ValueError(f'Tokenizer mismatch at {item["id"]}')
        if len(gguf_ids) > 8192:
            raise ValueError(f'Context overflow at {item["id"]}: {len(gguf_ids)}')
        started = time.monotonic()
        llm.reset()
        llm.eval(gguf_ids)
        scores = binary_scores(llm.scores[llm.n_tokens - 1], label_ids)
        raw = llm.detokenize([scores['greedy_token_id']]).decode('utf-8', errors='replace').strip()
        row = {k:item[k] for k in ('id','dataset','split','arm','seed','prompt_variant','prompt_sha256')}
        if 'label' in item:
            row['label'] = item['label']
        row.update(scores, raw=raw, response_valid=raw in ('0','1'), token_ids={'0':label_ids[0],'1':label_ids[1]}, prompt_tokens=len(gguf_ids), model_sha256=model_digest, model_repo=MODEL_REPO, model_repo_revision='local-sha256:'+model_digest, quantization='Q4_K_M', backend='llama-cpp-python', llama_cpp_python=llama_cpp_version, gpu=gpu, seed=42, elapsed_s=time.monotonic()-started, pilot_only=False, validation_only=True)
        f.write(json.dumps(row, ensure_ascii=False, allow_nan=False)+'\n')
        f.flush()
        os.fsync(f.fileno())
        done[item['id']] = row
        note(f'[{ix}/{len(rows)}] {item["id"]}: logprob_0={scores["logprob_0"]:.6f} logprob_1={scores["logprob_1"]:.6f} s={scores["s"]:.6f} raw={raw!r}')

status = {'status':'complete','expected':len(rows),'gpu':gpu,'all_layers_offloaded':all_layers_offloaded,'runtime_cap_seconds':MAX_RUNTIME_SECONDS,'expected_records':len(rows)}
status['records_written'] = sum(1 for line in records_path.read_text(encoding='utf-8').splitlines() if line.strip())
status['status'] = 'complete' if status['records_written'] == len(rows) else 'incomplete_or_timed_out'
(OUT / 'pilot_status.json').write_text(json.dumps(status, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
note(f'Validation output archive: {archive}')
